# Notebook 02 — Seeing the Map (PCA)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

We built 4-D word vectors and measured similarity — but we never *saw* the space. You can't
picture 4 directions at once. This notebook learns the one trick that works for *any* number
of directions, from our 4 up to the model's 384: **PCA**.

> **Analogy:** PCA is the angle you photograph a sculpture from to show the most detail. A
> sculpture has depth (3-D); a photo has 2. A bad angle hides what matters; a good angle
> reveals it. PCA finds the best angle automatically — the one where the data spreads out the
> most.

## Step 0 — Install the tools

Run the cell below once. It installs the three libraries this notebook uses: `numpy` (number crunching), `matplotlib` (drawing plots), and `scikit-learn` (the PCA tool). On Colab they may already be present, in which case it finishes almost instantly. Expect a short burst of install text and then the word `Ready.`.

In [ ]:
# Installs the three tools this notebook needs (skip the wait if already installed).
%pip install -q numpy matplotlib scikit-learn
print("Ready.")  # confirmation that the install step finished

## Step 1 — Imports

This cell loads the three libraries. Nothing visible happens beyond the line `Imports ready.`. If it errors, re-run the install cell above.

In [2]:
import numpy as np                       # arrays and math, used to hold the vectors
import matplotlib.pyplot as plt          # plotting library, draws the scatter plots
from sklearn.decomposition import PCA    # the PCA tool that flattens many directions to 2

print("Imports ready.")

Imports ready.


## Step 2 — The same ten words

Each notebook stands on its own, so we re-type the ten 4-D word vectors from Notebook 01.
The four directions are still `animal_ness`, `vehicle_ness`, `size`, `alive`. We also tag
each word with a category so the plot can colour them.

In [3]:
#                       animal  vehicle  size   alive
word_vectors = {                          # each word becomes four numbers (a 4-D vector)
    "cat":      [0.9,   0.0,   0.2,   0.9],
    "dog":      [0.9,   0.0,   0.3,   0.9],
    "lion":     [0.9,   0.0,   0.8,   0.9],
    "mouse":    [0.9,   0.0,   0.1,   0.9],
    "car":      [0.0,   0.9,   0.5,   0.7],
    "truck":    [0.0,   0.9,   0.9,   0.7],
    "bicycle":  [0.0,   0.8,   0.2,   0.5],
    "airplane": [0.0,   0.9,   1.0,   0.8],
    "tree":     [0.0,   0.0,   0.7,   0.6],
    "rock":     [0.0,   0.0,   0.5,   0.0],
}
category = {                              # label each word so the plot can colour it
    "cat": "animal", "dog": "animal", "lion": "animal", "mouse": "animal",
    "car": "vehicle", "truck": "vehicle", "bicycle": "vehicle", "airplane": "vehicle",
    "tree": "other", "rock": "other",
}
colour_map = {"animal": "tab:orange", "vehicle": "tab:blue", "other": "tab:green"}  # one colour per category

print(f"{len(word_vectors)} words across {len(set(category.values()))} categories.")  # quick sanity count

10 words across 3 categories.


## Step 3 — Approach A: just plot two of the directions

Because *we* named the directions, we can pick two and use them as the x- and y-axes. Let's
plot `animal_ness` (x) against `vehicle_ness` (y).

> This only works because we labelled the directions. With a real model's 384, nobody
> knows what each one means — so we'll need Approach B (PCA).

When you run the next cell, a scatter plot appears: animals bunch on the right (high animal_ness), vehicles bunch toward the top (high vehicle_ness), and `tree` and `rock` sit near the origin. What matters is which dots group together.

In [4]:
words = list(word_vectors.keys())                      # fixed word order, reused for every list below
xs = [word_vectors[w][0] for w in words]   # animal_ness  -> x-axis value for each word
ys = [word_vectors[w][1] for w in words]   # vehicle_ness -> y-axis value for each word
colours = [colour_map[category[w]] for w in words]     # colour each dot by its category

plt.figure(figsize=(7, 5))                             # start a new plot, 7x5 inches
plt.scatter(xs, ys, c=colours, s=140, edgecolor="black")  # draw one dot per word (s = dot size)
for x, y, label in zip(xs, ys, words):                 # loop over every point...
    plt.annotate(label, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=11)  # ...and write its name beside it
plt.xlabel("animal_ness  (direction 0)")               # name the x-axis
plt.ylabel("vehicle_ness  (direction 1)")              # name the y-axis
plt.title("Approach A — two hand-chosen directions")   # plot title
plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1)               # fix the axis range so dots have margin
plt.grid(True, linestyle="--", alpha=0.4)              # faint dashed grid to read positions
plt.tight_layout()                                     # trim extra whitespace around the plot
plt.show()                                             # render the figure

/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6740/2648367657.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()                                             # render the figure


### The limitation

We threw away `size` and `alive`. This plot can't tell that `tree` is bigger than `rock`.
For 4 directions that's survivable; for 384 it's hopeless. Enter PCA.

## Step 4 — Approach B: PCA squashes every direction down to 2

PCA's job: look at all the data, find the single direction where points spread out the most (call it **PC1**), then the next-best direction at right angles (**PC2**), and place every point on those two. The result uses information from *all four* original directions.

The next code cell does two things. First it runs PCA and prints the input shape (10 words by 4 directions), the output shape (10 words by 2 components), and the percentage of spread each component kept. Then a later cell draws the 2-D plot.

**PC1 and PC2 aren't labelled.** PCA just finds the angle with the most contrast. You can often squint and guess what they capture, but the model doesn't tell you.

> **Read clusters, never directions.** A PCA plot can come out **mirrored or rotated** on a different machine, or even on a re-run. Left versus right and up versus down carry no meaning. The only thing you can trust is **closeness**: which points sit near each other. If two runs disagree on which side a cluster is on but agree on what clusters together, both are correct.

In [5]:
# Activity: run PCA to squash the 4-D vectors down to 2-D.
matrix = np.array([word_vectors[w] for w in words])    # stack the vectors into a 10x4 grid (10 words, 4 directions)
print(f"Input shape: {matrix.shape}   (words, directions)")  # confirm it is 10 rows by 4 columns

pca = PCA(n_components=2)                               # create a PCA that will keep 2 output directions
pcs = pca.fit_transform(matrix)                        # fit = find the best angles; transform = place each word on them -> 10x2
print(f"Output shape: {pcs.shape}   (words, 2 principal components)")  # now 10 rows by 2 columns

var = pca.explained_variance_ratio_                    # how much of the spread each new direction kept
print(f"\nPC1 captured {var[0] * 100:.0f}% of the spread.")   # PC1 = the most-spread direction
print(f"PC2 captured {var[1] * 100:.0f}% of the spread.")     # PC2 = the next-best, at right angles
print(f"Together: {sum(var) * 100:.0f}%   (the rest is lost squashing 4-D to 2-D)")  # the leftover is what flattening discards

Input shape: (10, 4)   (words, directions)
Output shape: (10, 2)   (words, 2 principal components)

PC1 captured 65% of the spread.
PC2 captured 20% of the spread.
Together: 85%   (the rest is lost squashing 4-D to 2-D)


Now plot the 2-D result we just computed (the `pcs` array and the `var` percentages from the cell above). A scatter plot appears with PC1 on the x-axis and PC2 on the y-axis. Look at the clusters, and remember the picture may be flipped or turned compared to a classmate's. That is fine.

In [6]:
# Activity: plot the PCA result (uses pcs and var from the cell above).
plt.figure(figsize=(8, 5.5))                           # new plot canvas
plt.scatter(pcs[:, 0], pcs[:, 1], c=colours, s=140, edgecolor="black")  # column 0 = PC1 (x), column 1 = PC2 (y)
for (x, y), label in zip(pcs, words):                  # label each point with its word
    plt.annotate(label, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=11)
plt.xlabel(f"PC1  ({var[0] * 100:.0f}% of spread)")    # axis shows how much spread PC1 kept
plt.ylabel(f"PC2  ({var[1] * 100:.0f}% of spread)")    # axis shows how much spread PC2 kept
plt.title("Approach B — PCA from 4-D to 2-D (uses all directions)")
plt.grid(True, linestyle="--", alpha=0.4)              # faint grid
plt.tight_layout()
plt.show()

/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6740/1813005229.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### What to notice
1. Animals cluster on one side, vehicles on the other, `tree` and `rock` off on their own —
   the same grouping as before, but now the axes are *found*, not hand-picked.
2. Squint: PC1 probably captures animal-vs-vehicle (where most variation lives); PC2 maybe
   size or alive.
3. The %s matter: PC1 usually captures the most, PC2 the next. Together ~80–90% here, the
   rest lost to flattening.

## Step 5 — Predict, then verify: add three words

We add three new words with hand-picked numbers, then re-run PCA on the bigger set. Before
running, guess where each lands — near which existing words?

- `scooter` — a small wheeled thing
- `whale` — a huge ocean animal
- `flower` — alive, small, not an animal

Running the next cell rebuilds the data with these three words added, runs PCA again on all thirteen, and draws the plot. The three new words are drawn larger with a red border so you can spot them. Because PCA recomputes from scratch, the whole picture may shift or flip from the earlier plot; compare clusters, not exact spots.

In [7]:
#                                  animal  vehicle  size   alive
new_words = {                              # three extra words to test our predictions
    "scooter":   [0.0,   0.7,   0.3,   0.5],
    "whale":     [0.9,   0.0,   1.0,   0.9],
    "flower":    [0.0,   0.0,   0.1,   0.7],
}
new_category = {"scooter": "vehicle", "whale": "animal", "flower": "other"}  # categories for colouring

all_vectors = {**word_vectors, **new_words}            # merge old + new into one dictionary
all_categories = {**category, **new_category}          # merge the category labels too
all_words = list(all_vectors.keys())                   # combined word order
all_matrix = np.array([all_vectors[w] for w in all_words])  # stack all 13 vectors into a 13x4 grid
all_colours = [colour_map[all_categories[w]] for w in all_words]  # colour per word

pca2 = PCA(n_components=2)                              # a fresh PCA for the bigger set
pcs2 = pca2.fit_transform(all_matrix)                  # recompute the best 2 directions from scratch -> 13x2

plt.figure(figsize=(9, 6))                             # new plot canvas
for (x, y), word, c in zip(pcs2, all_words, all_colours):  # draw each word one at a time
    is_new = word in new_words                         # the three added words get a special look
    plt.scatter(x, y, c=c, s=200 if is_new else 140,   # new words drawn bigger
                edgecolor="red" if is_new else "black",  # new words get a red border
                linewidths=2 if is_new else 1)         # new words get a thicker border
    plt.annotate(word, (x, y), xytext=(6, 4), textcoords="offset points",
                 fontsize=12, fontweight="bold" if is_new else "normal")  # bold label for new words
plt.xlabel(f"PC1  ({pca2.explained_variance_ratio_[0] * 100:.0f}% of spread)")  # PC1 spread for this set
plt.ylabel(f"PC2  ({pca2.explained_variance_ratio_[1] * 100:.0f}% of spread)")  # PC2 spread for this set
plt.title("PCA with three new words (red border)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6740/18557952.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Did your guesses land? — and recap

- `scooter` should sit in the vehicle cluster (near `bicycle`); `whale` in the animal
  cluster (pulled toward `lion` by size); `flower` near `tree`.
- Adding words makes PCA recompute from scratch, so exact positions shift — don't expect
  identical coordinates. Clusters stay; coordinates wander.

**Recap**
- PCA finds the angle where data is most spread out, and flattens onto it.
- Read a PCA plot by its clusters; the axis %s tell you how faithful the 2-D picture is.
- Pick-two-directions works when you named them; PCA works always.

**Next (Notebook 03):** enough hand-built numbers — we hand real sentences to a real model
and get back real 384-dimensional embeddings, then use these exact tools on them.

## Practice — Your Turn

Two exercises to make PCA stick. Read the task, make your prediction, then run the answer cell to check it. Remember the one rule that never changes: read a PCA plot by its clusters. Which dots sit close together is the only thing you can trust. The plot may come out mirrored or turned compared to the earlier ones, and that is fine.

### Exercise 1 — Add three of your own words

Make a copy of the ten word vectors, then add three new words with sensible 4-number ratings on the same directions (`animal_ness`, `vehicle_ness`, `size`, `alive`). We will use `hamster`, `scooter`, and `boat`. Re-run PCA on the combined set of thirteen and plot the result.

Before you run it, predict: `hamster` is a small animal, `scooter` and `boat` are vehicles. Where should each one land? The two animals (`hamster` plus the existing ones) should group on one side, the vehicles on the other.

Watch the closeness, not the left or right. Your plot may look flipped from the one in Step 4, and that does not change the answer.

Try it yourself, then run the answer cell below.

In [8]:
# Answer
#                              animal  vehicle  size   alive
practice_new = {                          # three new words with hand-picked ratings
    "hamster":  [0.9,   0.0,   0.1,   0.9],   # small living animal
    "scooter":  [0.0,   0.7,   0.3,   0.5],   # small wheeled vehicle
    "boat":     [0.0,   0.8,   0.8,   0.6],   # large vehicle, not alive in the usual sense
}
practice_cat = {                          # category for each new word, used for colour
    "hamster": "animal", "scooter": "vehicle", "boat": "vehicle",
}

practice_vectors = {**word_vectors, **practice_new}      # COPY of the original plus the new words (original dict untouched)
practice_categories = {**category, **practice_cat}       # merged category labels
practice_words = list(practice_vectors.keys())           # combined word order
practice_matrix = np.array([practice_vectors[w] for w in practice_words])  # stack into a 13x4 grid
practice_colours = [colour_map[practice_categories[w]] for w in practice_words]  # one colour per word

practice_pca = PCA(n_components=2)                        # fresh PCA for this bigger set
practice_pcs = practice_pca.fit_transform(practice_matrix)  # find the best 2 directions and place each word -> 13x2
practice_var = practice_pca.explained_variance_ratio_    # how much spread each new direction kept

plt.figure(figsize=(9, 6))                               # new plot canvas
for (x, y), word, c in zip(practice_pcs, practice_words, practice_colours):  # draw each word one at a time
    is_new = word in practice_new                        # the three added words get a special look
    plt.scatter(x, y, c=c, s=200 if is_new else 140,     # new words drawn bigger
                edgecolor="red" if is_new else "black",  # new words get a red border
                linewidths=2 if is_new else 1)           # thicker border for new words
    plt.annotate(word, (x, y), xytext=(6, 4), textcoords="offset points",
                 fontsize=12, fontweight="bold" if is_new else "normal")  # label every point
plt.xlabel(f"PC1  ({practice_var[0] * 100:.0f}% of spread)")   # PC1 spread for this set
plt.ylabel(f"PC2  ({practice_var[1] * 100:.0f}% of spread)")   # PC2 spread for this set
plt.title("Exercise 1 — three new words added (red border)")   # plot title
plt.grid(True, linestyle="--", alpha=0.4)                # faint grid to read positions
plt.tight_layout()                                       # trim extra whitespace
plt.show()                                               # render the figure

print("hamster should sit with the animals; scooter and boat with the vehicles.")  # what to look for
print("Compare clusters, not exact spots. The picture may be flipped from Step 4.")

hamster should sit with the animals; scooter and boat with the vehicles.
Compare clusters, not exact spots. The picture may be flipped from Step 4.


/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6740/2237537754.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()                                               # render the figure


### Exercise 2 — Compare two directions by hand

PCA is not the only way to look. When you named the directions yourself, you can just pick two and plot them straight. Here we plot `size` (direction 2) on the x-axis and `alive` (direction 3) on the y-axis for all ten original words.

Before you run it, predict which pairs will sit closest. For example, `cat` and `dog` have almost the same size and aliveness, so they should land nearly on top of each other. `rock` has low aliveness, so it should drop to the bottom.

After the plot, the cell measures the straight-line distance between every pair and prints the three closest pairs. Closeness is the only readable thing on a scatter like this, so that is what we measure.

Try it yourself, then run the answer cell below.

In [9]:
# Answer
from itertools import combinations               # to list every pair of words

dim_x, dim_y = 2, 3                               # direction 2 = size, direction 3 = alive
ex_words = list(word_vectors.keys())             # the ten original words, unchanged
ex_xs = [word_vectors[w][dim_x] for w in ex_words]   # size value -> x for each word
ex_ys = [word_vectors[w][dim_y] for w in ex_words]   # alive value -> y for each word
ex_colours = [colour_map[category[w]] for w in ex_words]  # colour each dot by its category

plt.figure(figsize=(7, 5))                       # new plot canvas
plt.scatter(ex_xs, ex_ys, c=ex_colours, s=140, edgecolor="black")  # one dot per word
for x, y, label in zip(ex_xs, ex_ys, ex_words):  # loop over every point...
    plt.annotate(label, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=11)  # ...and name it
plt.xlabel("size  (direction 2)")                # name the x-axis
plt.ylabel("alive  (direction 3)")               # name the y-axis
plt.title("Exercise 2 — size vs alive (two hand-chosen directions)")  # plot title
plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1)         # fix axis range so dots have margin
plt.grid(True, linestyle="--", alpha=0.4)        # faint dashed grid
plt.tight_layout()                               # trim extra whitespace
plt.show()                                       # render the figure

# Measure the straight-line distance between every pair on these two directions.
pairs = []                                       # will hold (distance, wordA, wordB)
for a, b in combinations(ex_words, 2):           # every unordered pair of words
    va = np.array([word_vectors[a][dim_x], word_vectors[a][dim_y]])  # point A on (size, alive)
    vb = np.array([word_vectors[b][dim_x], word_vectors[b][dim_y]])  # point B on (size, alive)
    dist = float(np.linalg.norm(va - vb))        # straight-line distance between the two points
    pairs.append((dist, a, b))                   # remember it

pairs.sort()                                     # smallest distance first = closest pairs
print("Three closest pairs on size vs alive:")   # header
for dist, a, b in pairs[:3]:                     # show the top three
    print(f"  {a:>8} and {b:<8}  distance {dist:.2f}")  # closer pairs have smaller numbers
print("\nCloseness is the only readable thing here. Pairs with the smallest distance sit nearest.")

Three closest pairs on size vs alive:
       cat and dog       distance 0.10
       cat and mouse     distance 0.10
     truck and airplane  distance 0.14

Closeness is the only readable thing here. Pairs with the smallest distance sit nearest.


/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6740/3855803395.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()                                       # render the figure
